## Connect to Database and pull data

In [1]:
!pip install pymysql
!pip install boto3
!pip install sqlalchemy
!pip install python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 1.9 MB/s eta 0:00:00


In [2]:
# package for env variables in google colab: https://pypi.org/project/colab-env/
!pip install colab-env -qU
import colab_env

ImportError: colab-env only works in a Google Colab notebook

In [4]:
from colab_env import envvar_handler # modify env variables in var.env file in google drive

ImportError: colab-env only works in a Google Colab notebook

In [19]:
import pymysql
import sys
import boto3
import os

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [8]:
connection = pymysql.connect(
    host=MAS405_AWS_MY_DB_ROUSER_HOST,
    user=MAS405_AWS_MY_DB_ROUSER_USER,
    port=MAS405_AWS_MY_DB_ROUSER_PORT,
    passwd=MAS405_AWS_MY_DB_ROUSER_PW,
    database=MAS405_AWS_MY_DB_ROUSER_DBNAME
)

In [47]:
with connection.cursor() as cursor:
    cursor.execute("SHOW TABLES")
    tables = cursor.fetchall()

# Print table names
for table in tables:
    print(table[0])

SuperBowl_coinTosses
UC_salaries_data
google_elev_68sites
iris_data


In [ ]:
# from sqlalchemy import select
# stmt = select(user_table).where(user_table.c.name == "spongebob")
# print(stmt)

In [86]:
# years: 2010-2014
# 3024476 rows
# features: row_names	id	year	location	first.name	last.name	title	gross.pay	regular.pay	overtime.pay	other.pay
# location: DIVISION OF AGRICULTURE AND NATURAL RESOURCES (DANR)
# UCOP: university california office of the president
avg_loc_pay_df = pd.read_sql("SELECT location, AVG(`gross.pay`) FROM UC_salaries_data GROUP BY location", con=connection)
avg_loc_pay_df

<ipython-input-86-f645626e0120>:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  avg_loc_pay_df = pd.read_sql("SELECT location, AVG(`gross.pay`) FROM UC_salaries_data GROUP BY location", con=connection)


,location,AVG(`gross.pay`)
0,Santa Cruz,26480.185848
1,Berkeley,33418.287407
2,DANR,34808.033162
3,Davis,48795.614970
4,Irvine,44389.647911
5,Los Angeles,51069.148947
6,Merced,28558.416354
7,Riverside,29015.387886
8,San Diego,48018.481947
9,San Francisco,82919.020569


In [9]:
pd.read_sql("SELECT * FROM UC_salaries_data LIMIT 10", connection)

/Users/claytonchan/opt/anaconda3/lib/python3.9/site-packages/pandas/io/sql.py:761: UserWarning: pandas only support SQLAlchemy connectable(engine/connection) ordatabase string URI or sqlite3 DBAPI2 connectionother DBAPI2 objects are not tested, please consider using SQLAlchemy
  warnings.warn(


,row_names,id,year,location,first.name,last.name,title,gross.pay,regular.pay,overtime.pay,other.pay
0,1,1,2010,Santa Cruz,*****,*****,ASSISTANT PROFESSOR-ACAD YR,74511.10,69999.96,0.0,4511.14
1,2,2,2010,Berkeley,***********,***********,GRAD STDNT RES-FULL FEE REM,5369.42,5369.42,0.0,0.00
2,3,3,2010,Berkeley,***********,***********,TEACHING ASSISTANT - GSHIP,4802.00,4802.00,0.0,0.00
3,4,4,2010,Berkeley,***********,***********,GRAD STDNT RES- NO REMISSION,3409.01,3409.01,0.0,0.00
4,5,5,2010,Berkeley,***********,***********,GRAD STDNT RES-FULL FEE REM,11673.90,11673.90,0.0,0.00
5,6,6,2010,Berkeley,***********,***********,ASSISTANT II,1712.44,1698.76,0.0,13.68
6,7,7,2010,Berkeley,***********,***********,ASSISTANT I,1325.50,1325.50,0.0,0.00
7,8,8,2010,Berkeley,***********,***********,GRAD STDNT RES-FULL FEE REM,6861.94,6861.94,0.0,0.00
8,9,9,2010,Berkeley,***********,***********,ASSISTANT I,1232.00,1232.00,0.0,0.00
9,10,10,2010,Berkeley,***********,***********,READER - GSHIP,11067.81,11067.81,0.0,0.00


In [44]:
df = pd.read_sql("SELECT * FROM UC_salaries_data", connection)
df.head()

/Users/claytonchan/opt/anaconda3/lib/python3.9/site-packages/pandas/io/sql.py:761: UserWarning: pandas only support SQLAlchemy connectable(engine/connection) ordatabase string URI or sqlite3 DBAPI2 connectionother DBAPI2 objects are not tested, please consider using SQLAlchemy
  warnings.warn(


,row_names,id,year,location,first.name,last.name,title,gross.pay,regular.pay,overtime.pay,other.pay
0,1,1,2010,Santa Cruz,*****,*****,ASSISTANT PROFESSOR-ACAD YR,74511.10,69999.96,0.0,4511.14
1,2,2,2010,Berkeley,***********,***********,GRAD STDNT RES-FULL FEE REM,5369.42,5369.42,0.0,0.00
2,3,3,2010,Berkeley,***********,***********,TEACHING ASSISTANT - GSHIP,4802.00,4802.00,0.0,0.00
3,4,4,2010,Berkeley,***********,***********,GRAD STDNT RES- NO REMISSION,3409.01,3409.01,0.0,0.00
4,5,5,2010,Berkeley,***********,***********,GRAD STDNT RES-FULL FEE REM,11673.90,11673.90,0.0,0.00


In [45]:
df = df.dropna()
df

,row_names,id,year,location,first.name,last.name,title,gross.pay,regular.pay,overtime.pay,other.pay
0,1,1,2010,Santa Cruz,*****,*****,ASSISTANT PROFESSOR-ACAD YR,74511.10,69999.96,0.0,4511.14
1,2,2,2010,Berkeley,***********,***********,GRAD STDNT RES-FULL FEE REM,5369.42,5369.42,0.0,0.00
2,3,3,2010,Berkeley,***********,***********,TEACHING ASSISTANT - GSHIP,4802.00,4802.00,0.0,0.00
3,4,4,2010,Berkeley,***********,***********,GRAD STDNT RES- NO REMISSION,3409.01,3409.01,0.0,0.00
4,5,5,2010,Berkeley,***********,***********,GRAD STDNT RES-FULL FEE REM,11673.90,11673.90,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...
3024470,3024471,210097,2020,Riverside,BARA,MBOUP,MUSICIAN 2,4747.00,4747.00,0.0,0.00
3024471,3024472,210098,2020,Irvine,EPHANTUS,MBUGUA,REG RESP THER 1,38428.00,31437.00,713.0,6278.00
3024472,3024473,210099,2020,Los Angeles,PAULINE,MBUGUA,BLANK AST 2 PD,41089.00,39061.00,1953.0,75.00
3024473,3024474,210100,2020,Irvine,BERNADETTE,MBURU,PSYCHIATRIC TCHN SR,55677.00,52624.00,40.0,3013.00


In [84]:
pat = r"""(?ix)                     # (?i)=ignore-case  (?x)=verbose mode
    \b(?:                               # word-start, then one of…
        prof(?:essors?\b|\.?\b)         #  "professor" / "professors" / "prof" / "prof."
      | lect(?:urer?s?\b|\b)            #  "lecturer(s)" / "lect"
    )
"""
prof_rows = df[df['title'].str.contains(pat,case = False,na = False)]

In [85]:
prof_rows

,row_names,id,year,location,first.name,last.name,title,gross.pay,regular.pay,overtime.pay,other.pay
0,1,1,2010,Santa Cruz,*****,*****,ASSISTANT PROFESSOR-ACAD YR,74511.10,69999.96,0.0,4511.14
20,21,21,2010,Berkeley,***********,***********,LECTURER - ACADEMIC YEAR,30364.71,25454.75,0.0,4909.96
94,95,95,2010,Berkeley,***********,***********,LECTURER - ACADEMIC YEAR,29248.46,29248.46,0.0,0.00
175,176,176,2010,Berkeley,***********,***********,LECTURER - ACADEMIC YEAR,27258.69,27058.69,0.0,200.00
178,179,179,2010,Berkeley,***********,***********,LECTURER - ACADEMIC YEAR,28679.78,28679.78,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...
3024446,3024447,210073,2020,Los Angeles,ALEXANDER,MAZZAFERRO,ASST PROF-AY,39417.00,39417.00,0.0,0.00
3024450,3024451,210077,2020,San Diego,WILLIAM,MAZZEI,HS CLIN PROF-HCOMP,218546.00,163606.00,0.0,54940.00
3024455,3024456,210082,2020,Los Angeles,JOHN,MAZZIOTTA,PROF-HCOMP,1035172.00,695172.00,0.0,340000.00
3024456,3024457,210083,2020,Los Angeles,MAURIZIO,MAZZOCCO,ASSOC PROF-AY-B/E/E,283000.00,283000.00,0.0,0.00


In [81]:
titles = df["title"].unique().tolist()
titles = [i for i in titles if "LECT" in i]
titles

['LECTURER - ACADEMIC YEAR',
 'LECTURER IN SUMMER SESSION',
 'LECTURER - ACADEMIC YEAR - 1/9',
 'ELECTRONIC COMM SPECIALIST 3',
 'LECTURER-AY-1/9-CONTINUING',
 'LECTURER-AY-CONTINUING APPT.',
 'TECHNICIAN,ELECTRONICS,TRAINEE',
 'LECTURER - FISCAL YEAR',
 'OFFICER, ELECTED, STUDENT GOVT',
 'TECHNICIAN, ELECTRONICS, SR',
 'COLLECTIONS REPRESENTATIVE, SR',
 'SR LECT W/SEC EMPL-ACADEMIC YR',
 'LECTURER-MISC/PART-TIME',
 'LECTURER W/SEC EMPL-ACAD YR',
 'ELECTRICIAN',
 'INTELLECTUAL PROPERTY OFFICR 4',
 'LECTURER-FY-CONTINUING APPT.',
 'TECHNICIAN, ELECTRONICS',
 'TECHNICIAN, ELECTROCARDIOGRAPH',
 'TECHNICIAN, ELECTRONICS, PRIN',
 'COLLECTIONS REPRESENTATIVE',
 'TECHNICIAN, ELECTROCARDIOG, SR',
 'LECTURER-PSOE-ACAD YR-100%',
 'INTELLECTUAL PROPERTY OFFICR 1',
 'SR LECT W/SEC EMPL - FISCAL YR',
 'SR. LECTURER-AY-CONTINUING',
 'HIGH VOLT ELECTRICIAN',
 'ELECTRONIC COMM SPEC 4',
 'ELECTRICIAN, SUPERVISING',
 'ARTS AND LECTURES MANAGER',
 'ELECTRICIAN, MARINE, UTILITY',
 'ELECTRICIAN, APPRENTICE

In [83]:
"LECT" in "ELECTRICIAN"

True

In [26]:
[i for i in titles if "PROFESSIONAL" not in i and "PROFL" not in i]

['ASSISTANT PROFESSOR-ACAD YR',
 'ACT PROFESSOR-LAW SCHOOL SCALE',
 'ASSISTANT PROFESSOR-SFT-VM',
 'HS ASSISTANT CLIN PROF-HCOMP',
 'ASST PROF OF CLIN_____-HCOMP',
 'ASST PROF OF CLINICAL_____-FY',
 'HS ASST CLIN PROFESSOR-FY',
 'VST ASST PROFESSOR -FISCAL YR',
 'ASST CLINICAL PROFESSOR-VOL',
 'VISITING ASSISTANT PROF-HCOMP',
 'ASST PROF IN RESIDENCE - FY',
 'ASSISTANT PROF IN RES-HCOMP',
 'ASSISTANT ADJUNCT PROF-HCOMP',
 'ASST ADJUNCT PROF-MEDCOMP-A',
 'PROFESSOR - ACADEMIC YEAR',
 'ASSOCIATE PROF IN RES-HCOMP',
 'PROFESSOR-HCOMP',
 'ASST PROF-ACAD YR-BUS/ECON/ENG',
 'HS CLIN PROFESSOR-FISCAL YR',
 'PROFESSOR-ACAD YR-BUS/ECON/ENG',
 'ASSOC PROF OF CLIN_____-HCOMP',
 'PROFESSOR-ACAD YR-RECALLED',
 'HS ASSOCIATE CLIN PROF-HCOMP',
 'PROFESSOR-LAW SCHOOL SCALE',
 'PROFESSOR RECALLED-ACAD YR-1/9',
 'VIS ASST PROFESSOR-GENCOMP',
 'ASSOCIATE PROFESSOR-ACAD YR',
 'PROFESSOR OF CLINICAL___-HCOMP',
 'ASSOCIATE PROFESSOR-HCOMP',
 'ASSOCIATE ADJUNCT PROF-HCOMP',
 'ASSOC PROF-AY-BUS/ECON/ENG',
 'V

In [86]:
def title_assign(x):
    if "PROFL" in x or "PROFESSIONAL" in x:
        return "OTHER"
    elif "VST" in x and "ASST" in x:
        return "VST ASST"
    elif "VST" in x and "ASSOC" in x:
        return "VST ASSOC"
    elif "VST" in x:
        return "VST PROF"
    elif "ADJ" in x and "ASST" in x:
        return "ADJ ASST"
    elif ("ASOC" in x or "ASSOC" in x) and "ADJ" in x:
        return "ADJ ASOC"
    elif "ADJ" in x:
        return "ADJ PROF"
    elif ("ASST" in x or "ASSIST" in x) and "PROF" in x:
        return "ASST PROF"
    elif "ASSOC" in x and "PROF" in x:
        return "ASSOC PROF"
    elif "PROF" in x:
        return "PROF"
    elif "SR" in x and "LECT" in x:
        return "SR LECT"
    elif "LECT" in x:
        return "LECT"
    else:
        return "OTHER"

In [87]:
prof_rows["title"] = prof_rows["title"].apply(title_assign)

/var/folders/py/yn4jz9f166sd76n0j33k5ny00000gn/T/ipykernel_30120/1405741927.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prof_rows["title"] = prof_rows["title"].apply(title_assign)


In [88]:
prof_rows[prof_rows["title"]!="OTHER"]

,row_names,id,year,location,first.name,last.name,title,gross.pay,regular.pay,overtime.pay,other.pay
0,1,1,2010,Santa Cruz,*****,*****,ASST PROF,74511.10,69999.96,0.0,4511.14
20,21,21,2010,Berkeley,***********,***********,LECT,30364.71,25454.75,0.0,4909.96
94,95,95,2010,Berkeley,***********,***********,LECT,29248.46,29248.46,0.0,0.00
175,176,176,2010,Berkeley,***********,***********,LECT,27258.69,27058.69,0.0,200.00
178,179,179,2010,Berkeley,***********,***********,LECT,28679.78,28679.78,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...
3024446,3024447,210073,2020,Los Angeles,ALEXANDER,MAZZAFERRO,ASST PROF,39417.00,39417.00,0.0,0.00
3024450,3024451,210077,2020,San Diego,WILLIAM,MAZZEI,PROF,218546.00,163606.00,0.0,54940.00
3024455,3024456,210082,2020,Los Angeles,JOHN,MAZZIOTTA,PROF,1035172.00,695172.00,0.0,340000.00
3024456,3024457,210083,2020,Los Angeles,MAURIZIO,MAZZOCCO,ASSOC PROF,283000.00,283000.00,0.0,0.00


In [89]:
prof_rows.groupby("title")["gross.pay"].mean().sort_values()

title
VST ASSOC      18720.152473
VST ASST       20629.142353
VST PROF       28685.205937
LECT           32005.976347
ADJ ASST       83534.351612
ADJ ASOC       97817.998927
ADJ PROF       98626.052368
SR LECT       118420.398696
ASST PROF     150323.815197
ASSOC PROF    171999.885210
PROF          217624.153142
Name: gross.pay, dtype: float64

In [64]:
prof_rows.groupby("location")["gross.pay"].mean().sort_values()

location
UCOP                        20223.674186
DANR                        25276.898667
Hastings                    61655.841121
Hastings College Of Law     62853.018868
Merced                     113714.270824
Riverside                  118308.432540
Santa Cruz                 123314.845434
Santa Barbara              141446.017037
Berkeley                   150359.078108
Davis                      172797.047093
Irvine                     177607.196310
San Diego                  197651.592376
San Francisco              207881.533423
Los Angeles                209347.684015
Name: gross.pay, dtype: float64

In [71]:
np.isin(prof_rows["title"],["PROF"])

array([False,  True, False, ...,  True, False,  True])

In [82]:
pd.DataFrame(prof_rows.groupby(["year","title"])["gross.pay"].mean())

gross.pay
year title                    
2010 ADJ ASOC     95952.950491
     ADJ ASST     38646.620972
     ADJ PROF     96407.767079
     ASSOC PROF  138597.745982
     ASST PROF   126382.080895
...                        ...
2020 ADJ PROF     96265.327338
     ASSOC PROF  214972.863824
     ASST PROF   179259.301004
     PROF        269055.442667
     VST ASSOC    12444.000000

[96 rows x 1 columns]

In [79]:
prof_rows[np.isin(prof_rows['first.name'],["FREDERIC"])]

,row_names,id,year,location,first.name,last.name,title,gross.pay,regular.pay,overtime.pay,other.pay
879385,879386,105396,2013,San Francisco,FREDERIC,AMBROGGI,ADJ ASST,29750.0,29750.0,0.0,0.0
1151183,1151184,108752,2014,San Francisco,FREDERIC,AMBROGGI,ADJ ASST,81150.0,77069.0,0.0,4081.0
1297543,1297544,255112,2014,Berkeley,FREDERIC,THEUNISSEN,PROF,157760.0,118558.0,0.0,39202.0
1427704,1427705,110017,2015,San Francisco,FREDERIC,AMBROGGI,ADJ ASST,68361.0,49586.0,0.0,18775.0
1578497,1578498,260810,2015,Berkeley,FREDERIC,THEUNISSEN,PROF,163924.0,121323.0,0.0,42601.0
1868897,1868898,269696,2016,Berkeley,FREDERIC,THEUNISSEN,PROF,177105.0,137843.0,0.0,39262.0
1874686,1874687,275485,2016,San Francisco,FREDERIC,VAN GOOL,ADJ ASST,86488.0,86488.0,0.0,0.0
2065236,2065237,174894,2017,Los Angeles,FREDERIC,GRILLOT,PROF,22665.0,22665.0,0.0,0.0
2166451,2166452,276109,2017,Berkeley,FREDERIC,THEUNISSEN,PROF,165819.0,147075.0,0.0,18744.0
2172396,2172397,282054,2017,San Francisco,FREDERIC,VAN GOOL,ADJ ASST,90183.0,90183.0,0.0,0.0
